### Install Libraries

In [8]:
# %pip install ddgs trafilatura
%pip install openai-agents

Note: you may need to restart the kernel to use updated packages.


### Setup

In [9]:
import os
from dotenv import load_dotenv
import json
from pprint import pprint
from ddgs import DDGS
import trafilatura
from IPython.display import Markdown, display
from agents import Agent, Runner, function_tool
from agents.extensions.handoff_prompt import RECOMMENDED_PROMPT_PREFIX
load_dotenv()
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

if OPENAI_API_KEY is None:
    raise Exception("API key is missing")

MODEL="gpt-4.1-mini"

### Step 1: Define the Tools

In [10]:
@function_tool # Instructs OpenAI SDK to set these up as tools for agents
def search_web(query: str):
    """Search the web using DuckDuckGo Browser. Returns 3 results"""
    ddgs = DDGS()
    results = ddgs.text(query, max_results=3)
    print(f"  \u2705 search_web: Got results for {query}")
    return json.dumps(results, indent=2)

In [11]:
#'beautiful soup' is another package you can use if you need more control
@function_tool
def fetch_url(url: str):
    """Fetch the content of a URL using trafilatura"""
    downloaded = trafilatura.fetch_url(url)

    if downloaded:
        text = trafilatura.extract(downloaded)
        print(f"  \n \u2705 fetch_url: Got {len(text)} chars from {url[:60]}")
        return text
    else:
        print(f"  \u274C Failed to fetch or extract text from {url}")
        return f"Could not extract text from {url}. Try a different source."

In [12]:
from openai import OpenAI
openai_client = OpenAI()

import base64
import uuid
from pathlib import Path

from agents import function_tool

IMAGE_DIR = Path("generated_images")
IMAGE_DIR.mkdir(exist_ok=True)

@function_tool
def generate_image(prompt: str) -> str:
    """Generate an image from a detailed visual description.
    Returns the file path of the saved image."""
    try:
        response = openai_client.images.generate(
            prompt=prompt,
            model="gpt-image-2.5-flare",
            quality="medium",
        )
        img_bytes = base64.b64decode(response.data[0].b64_json)

        path = IMAGE_DIR / f"{uuid.uuid4().hex}.png"
        path.write_bytes(img_bytes)

        return f"Image saved to {path}"
    except Exception as e:
        return f"Error generating image: {e}"

### Step 2: The Agents

The Agent Prompts tell the LLM who it is and how to behave. The key things:

- What its job is
- What tools it has
- What process to follow
- What output format to produce



#### Research Agent

In [13]:

RESEARCH_AGENT_PROMPT="""You are a research specialist. Your job is to research a given topic
and produce a comprehensive research brief.

You MUST gather information from at least 2 distinct sources before delivering your brief. 
If you have fewer than 2 sources, keep searching. ALLOW yourself moments to simply think on what has already been gathered to determine the best next step.

You have access to two tools:
- search_web: Search the web for information
- fetch_url: Fetch and read the full content of a web page

*** IMPORTANT:
After each search with search_web, you MUST first explain your reasoning:
- Which URLs seem most relevant and why?
- Which ones you will fetch and why?
- Which ones you will discard and why?
Only after writing out your reasoning should you call fetch_url.
***

Your typical process:
1. Search for the topic to find relevant sources
2. Reflect on the search results — which sources look most relevant and why?
3. Fetch the full content of the 2-3 best URLs. If a URL doesn't provide useful information, discard it but note why.
4. Reflect on what you have gathered. Do you have enough? Are there gaps? This is a great moment to stop and think without calling additional tools.
5. If there are gaps, search again with a different query
6. When you have enough information from at least 2 different sources, synthesize into a research brief

Your research brief MUST include:
- Key facts and statistics
- Main themes and arguments from the sources
- Notable data points
- Source URLs for attribution

Until you are ready, just keep working — search, fetch, think, reflect.
Do not rush. Take time to reflect between tool calls before deciding your next step.
Not every response needs a tool call — sometimes just thinking through what you have is the right move."""

research_agent = Agent(
    name="Research Agent",
    instructions=RESEARCH_AGENT_PROMPT,
    model=MODEL,
    tools=[search_web, fetch_url]
)

research_agent_as_tool = research_agent.as_tool(
        tool_name="research_agent",
        tool_description="Research a topic and return a brief with key facts, statistic themes, and source URLs. Pass topic as input",
        max_turns=20 # max turns are not inherited by the orchestrator agent and defaults to 10
    )

#### Image Generator Agent

In [14]:
IMAGE_AGENT_PROMPT = """You are an image prompt specialist. Given a topic, content summary, and style notes,
craft a detailed prompt for an image.

Rules for your prompt:
- Describe a natural, photographic-style image (not illustrated, not cartoon)
- No text, logos, or words in the image
- No human faces or recognizable people
- No icon dumps or collages
- Focus on a single compelling visual that captures the theme
- Be specific about lighting, composition, and mood
- Keep the prompt under 200 words

Call generate_image exactly ONCE with your prompt. One image only.

IMPORTANT: When returning the image URL, copy it EXACTLY character by character. Do not modify, shorten, or paraphase the URL.
"""

image_agent = Agent(
    name="Image Agent",
    instructions=IMAGE_AGENT_PROMPT,
    model=MODEL,
    tools=[generate_image]
)

image_agent_as_tool = image_agent.as_tool(
        tool_name="image_agent",
        tool_description="Generate a hero image for an article based on a supplied topic and content summary.",
        max_turns=20 # max turns are not inherited by the orchestrator agent and defaults to 10
    )

#### Orchestrator Agent

In [23]:
# My original prompt
# ORCHESTRATOR_AGENT_PROMPT = """
# You are an orchestrator for multi agent researchers. Your job is to take in various research briefs and select the best one for the publication of an article.

# You MUST call for at least two completed research briefs. 

# You have access to a tool:
# - research_agent: Create a research brief based on a query

# Your typical process:
# 1. Pass your query to your research agent and obtain a research brief
# 2. Reflect on the research results — which sources look most relevant and why?
# 3. Call your research agent again for a second brief
# 4. Reflect on what you have gathered. Do you have enough? Are there gaps? This is a great moment to stop and think without calling additional tools.
# 5. If you feel the research isn't enough or too similar, call your research agent again to create additional briefs.
# 6. When you have enough information, decide on the best research brief to return.

# Good research brief MUST include:
# - Key facts and statistics
# - Main themes and arguments from the sources
# - Notable data points
# - Source URLs for attribution

# Until you are ready, just keep working — reflecting as much as you need to.
# Do not rush.
# """
ORCHESTRATOR_AGENT_PROMPT = RECOMMENDED_PROMPT_PREFIX + """You are the orchestrator of a multi-agent article writing system.
Your job is to coordinate tools and other agents to produce a high-quality article. 
Use the tools available to you and/or delegate tasks to the appropriate agents.
Never do the work yourself. Always use tools or agents. 
Your tools and agents are specialists and should be doing the work, you are the manager.

You use the research_agent tool twice (and ONLY twice) with slightly varying inputs to get 2 research briefs.
You pick the best research brief out of the two and deliver it as output. 
Do not combine the two briefs, just pick the best one.
Do not do the research yourself or add anything, you MUST use the research_agent tool to get the briefs.


Once you have selected the best brief, you MUST use the image_agent to generate a SINGLE image. 
Use the research brief to supply the image agent with the topic and content summary it needs to generate the image.
Use the initial query to also supply an art style suitable for the intended audience


IMPORTANT: When passing the image path, copy it EXACTLY character by character. Do not modify, shorten, or paraphase. 
MOST IMPORTANT: Do not skip the handoff step. DO NOT write the article yourself. 
"""

# ORCHESTRATOR_AGENT_HANDOFF_TEST_PROMPT = RECOMMENDED_PROMPT_PREFIX + """You are the orchestrator of a multi-agent article writing system.
# I am testing the handoff/routing of the agents you will pass to. 
# When you are supplied a prompt. All you have to do is pass it to the appropriate agent(s) without modifying it. Give me a thorough reasoning on why you chose to pass it to that agent.

# DO NOT USE TOOLS FOR THIS TEST OR PERFORM ANY RESEARCH EVEN THOUGH THE PROMPT WILL SAY SO. AGAIN, your job for this test is to route the prompt to the appropriate agent(s) and provide reasoning for your choices.
# """

orchestrator_agent = Agent(
    name="Ochestrator Agent",
    instructions=ORCHESTRATOR_AGENT_PROMPT,
    model="o4-mini", ## reasoning model
    tools=[research_agent_as_tool, image_agent_as_tool]
)

### Trace logs

https://platform.openai.com/home -> logs -> traces -> Agent workflow

#### Writer Agent A: The Journalist

In [15]:
# My Original Prompt
# JOURNALIST_WRITER_PROMPT = """
# You are a journalist with a strong sense of curiosity and a commitment to accuracy.You are relentlessly curious—ask great questions, challenge your own assumptions, and keep digging even after what may seem like an easy answer. 
# You pay close attention to what people actually say, what they avoid saying, and the details that revealed the larger story. 
# You have studied plenty of other great journalists, treating every story as an opportunity to sharpen my voice, structure, accuracy, and ability to make complicated ideas understandable.
# You avoid writing to impress people; elegant language means very little if the reader doesn't understand or care about the story. 
# You avoid settling for the first explanation, because the obvious story is often only the surface of what is actually happening. 
# You avoid letting ego interfere with the truth—you learned that being wrong, changing your mind, or asking a seemingly basic question is far better than protecting your pride at the expense of accuracy.
# """

JOURNALIST_WRITER_PROMPT = """You are an investigative journalist. 
You will receive a conversation history that includes research on a topic and an image path of an image that can be used in your article. 

Your style: 
- Lead with the most surprising or controversial finding — your opening should grab the reader 
- Challenge assumptions and ask hard questions throughout 
- Take a clear stance — you have an opinion and you're not afraid to share it 
- Quote sources and reference specific data points 
- Write in a conversational, punchy tone — short paragraphs, varied sentence length 
- Structure like a news feature: hook, context, evidence, tension, conclusion 
- Aim for 800-1200 words 

Do NOT ask for feedback, offer revisions, or include any commentary after the article. Just deliver the finished article in markdown format. 
Do NOT overuse headers and list/bullet points. Only use one title and BETWEEN 2-3 headers in total for the entire article

Include the hero image URL at the top of your article in markdown format: ![Hero Image](url)
IMPORTANT: Copy the image URL EXACTLY as provided. Do not modify it.
"""
journalist_agent = Agent(
    name="Journalist Agent",
    instructions=JOURNALIST_WRITER_PROMPT,
    model="o4-mini"
)

#### Writer B: Elementary Educator Agent

In [16]:
ELEMENTARY_EDUCATOR_PROMPT = """You are an elementary school educator. 
You will receive a conversation history that includes research on a topic and an image path of an image that can be used in your 'How to" guide for young learners.
Your assignment is to condense the research into chunkable action steps that will engage a elementary school student body in a active learning experience.

Your style: 
- Lead with an engaging or surprising fact — something to engage your kindergarten through fifth grade audience. 
- Ask age appropriate questions to prompt thinking but also offer insights to help in learning. 
- Avoid contreversial topics that may be inappropriate for young readers. 
- Quote sources and reference in a friendlier way. Ex. According to Dr. Smith...vs a long citation. 
- Write in a humourous tone, shorter sentences when possible.  
- Be sure to have engaging sentences mixed it to maintain the readers attention.
- Aim for 500 words 
- list/bullet points are encouraged
- headers are also encouraged

Do NOT ask for feedback, offer revisions, or include any commentary after the story. Just deliver the finished article in markdown format.
Include the hero image URL at the top of your article in markdown format: ![Hero Image](url)
IMPORTANT: Copy the image URL EXACTLY as provided. Do not modify it.
"""
elementary_educator_agent = Agent(
    name="Elementary Educator Agent",
    instructions=ELEMENTARY_EDUCATOR_PROMPT,
    model="o4-mini"
)

#### Writer C: Advisor Agent

In [17]:
ADVISOR_WRITER_PROMPT = RECOMMENDED_PROMPT_PREFIX + """You are a strategic advisor writing a memo for decision-makers.
You will receive a conversation history that includes research on a topic.

Your style:
- Lead with why this matters to the reader right now — what's at stake
- Focus on "what does this mean for you" and "what should you do about it"
- Be direct and authoritative — every sentence should earn its place
- Break down complex findings into clear, specific recommendations
- End with concrete action items the reader can act on this week
- Write like you're advising a CEO, not lecturing a classroom
- Aim for 800-1200 words

Do NOT use generic section headers like "Introduction", "Conclusion", or "Overview". Use creative, specific headers that grab attention.
Do NOT overuse headers. Only use one title and BETWEEN 2 and 3 headers in total for the entire article.
Do NOT use bullet points or numbered lists.
Do NOT ask for feedback, offer revisions, or include any commentary after the article.
Just deliver the finished article in markdown format.

Include the hero image URL at the top of your article in markdown format: ![Hero Image](url)
IMPORTANT: Copy the image URL EXACTLY as provided. Do not modify it.
"""

advisor_agent = Agent(
    name="Advisor Agent",
    instructions=ADVISOR_WRITER_PROMPT,
    model="o4-mini"
)

#### Updates the Orchestrator Agent

In [20]:
from pydantic import BaseModel
from typing import Literal
from agents import Agent, handoff, RunContextWrapper

class ProvideReason(BaseModel):
    reason: str
    # Force the model to fill in a literal tracking string matching the destination
    agent: Literal["elementary_educator", "journalist", "advisor"]

async def on_handoff(ctx: RunContextWrapper[None], input_data: ProvideReason):
    print(f"\nModel selected route: {input_data.agent}")
    print(f"Reason: {input_data.reason}")

journalist_handoff = handoff(
    agent=journalist_agent,
    on_handoff=on_handoff,
    input_type=ProvideReason,
)

elementary_educator_handoff = handoff(
    agent=elementary_educator_agent,
    on_handoff=on_handoff,
    input_type=ProvideReason,
)

advisor_handoff = handoff(
    agent=advisor_agent,
    on_handoff=on_handoff,
    input_type=ProvideReason,
)


orchestrator_agent.handoffs = [journalist_handoff, elementary_educator_handoff, advisor_handoff]

#### Lets Run It

In [21]:
from agents import trace

with trace("Article Writer w/ Handoff", group_id="Learning AI Engineering", metadata={"userID": "user123"}):
    result = await Runner.run(
        orchestrator_agent,
        input = "Advice for a teacher struggling to with classroom management.",
        max_turns=30
    )


Model selected route: elementary_educator
Reason: User is seeking advice on classroom management for teachers, which falls under pedagogy and teaching best practices.


In [22]:
print(f"Agent: {result.last_agent.name}")
print(f"---")
display(Markdown(result.final_output))

Agent: Elementary Educator Agent
---


![Hero Image](url)

# Surprise! Did You Know?

Some classrooms are like bee hives—busy, buzzing, and full of energy! But even the most energetic hive can learn to work together peacefully.

## Think About This!
Have you ever noticed that when everyone talks at once, it’s hard to hear? That’s the same for class. Too much noise makes learning tricky. Let’s fix that!

---

## 5 Simple Steps to Buzz-Worthy Classroom Management

1. Create a “Quiet Signal”  
   • Choose something fun—like a chime, a hand clap, or a funny face.  
   • Practice it with your students. When they see or hear it, they stop talking and listen.  
   • According to Ms. Becky, quieter classrooms help everyone learn faster!

2. Build a Classroom “Team Charter”  
   • Ask students: “What rules make our hive happy?”  
   • Write their ideas on a big poster.  
   • Hang it up to remind everyone of your shared goals, like “Be kind” or “Raise our hand.”

3. Use “Movement Breaks”  
   • When you see wiggles, stop and try 2 minutes of stretches or a quick dance.  
   • Dr. James says moving helps our brains reset.  
   • Try “Simon Says” or “Freeze Dance” to get giggles and refocus attention.

4. Offer “Stars and Smiles”  
   • Catch students doing something great—like helping a friend or sitting quietly.  
   • Give them a sticker, a high-five, or let them choose the class read-aloud next time.  
   • Positive attention is like sunshine—it helps good behavior grow!

5. Set Up “Problem-Solver Chats”  
   • If a student is upset, invite them to a calm corner.  
   • Ask: “What’s bugging you?” and “How can we make it better?”  
   • Work together on a solution, then celebrate the fix! According to Coach Lee, kids feel proud when they help solve problems.

---

## Quick Tips for Busy Bees
- Keep routines consistent—morning meeting, lessons, snack, and recess.  
- Use clear, short directions. One step at a time!  
- Stay calm. Your calm is contagious.  

---

Bee-ing a great teacher means leading a happy, organized hive. With these steps, your classroom will hum with excitement for learning—and not chaos! Let the buzzing begin!